In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import pandas as pd
from sklearn.feature_selection import mutual_info_classif             
import operator
import matplotlib.pyplot as plt
import numpy as np  
import pickle
import os
from huggingface_hub import login
from llama_index import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    ServiceContext,
)
from llama_index.node_parser import SentenceSplitter
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.embeddings.langchain import LangchainEmbedding
from llama_index.llms import HuggingFaceLLM
from transformers import AutoModelForCausalLM, AutoTokenizer
import faiss
from huggingface_hub import login

login(token="")


In [ ]:
from llama_index.schema import Document
import re
from langchain.embeddings import HuggingFaceEmbeddings


def clean_text(doc: Document) -> Document:
    # Remove headings like 2.2.3. or 1.1
    cleaned = re.sub(r"\b\d+(\.\d+)+\b", "", doc.text)
    return Document(text=cleaned, metadata=doc.metadata)

os.makedirs("pdfs", exist_ok=True)
documents_raw = SimpleDirectoryReader(input_dir="pdfs").load_data()

for doc in documents_raw:
    if not hasattr(doc, "metadata") or doc.metadata is None:
        doc.metadata = {}
    # Attach a name for later reference
    if hasattr(doc, "file_path"):
        doc.metadata["name"] = os.path.basename(doc.file_path)
    elif "file_path" in getattr(doc, "metadata", {}):
        doc.metadata["name"] = os.path.basename(doc.metadata["file_path"])
    else:
        doc.metadata["name"] = "Unknown document"

documents = [clean_text(doc) for doc in documents_raw]
print(f"Loaded and cleaned {len(documents)} documents.")

### 2. CHUNK DOCUMENTS, KEEP DOC NAME IN METADATA ###

parser = SentenceSplitter(chunk_size=256, chunk_overlap=20)
nodes = []
for doc in documents:
    doc_nodes = parser.get_nodes_from_documents([doc])
    for node in doc_nodes:
        node.metadata["doc_name"] = doc.metadata.get("name", "Unknown document")
    nodes.extend(doc_nodes)

### 3. CREATE VECTOR INDEX WITH EMBEDDINGS ###

embed_model = LangchainEmbedding(
    
    HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


)
faiss_index = faiss.IndexFlatL2(384)
vector_store = FaissVectorStore(faiss_index=faiss_index)




Loaded and cleaned 68 documents.


/tmp/ipykernel_4166829/2391901103.py:42: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "1,3"
llm = HuggingFaceLLM(
    model_name="hugging-quants/Meta-Llama-3.1-70B-Instruct-AWQ-INT4",
    tokenizer_name="hugging-quants/Meta-Llama-3.1-70B-Instruct-AWQ-INT4",
    context_window=5800,
    #max_new_tokens=800,
    max_new_tokens=2000,
    generate_kwargs={"temperature": 0.0,
    "do_sample": False,
    #"top_p": 0.9,
    #"repetition_penalty": 1.2,
    #"eos_token_id": 2,
    #"pad_token_id":2,
    },
    device_map="auto",
    tokenizer_kwargs={"use_fast": True},
    model_kwargs={"torch_dtype": "auto"},
)
service_context = ServiceContext.from_defaults(
    embed_model=embed_model,
    llm=llm
)
index = VectorStoreIndex(nodes, vector_store=vector_store, service_context=service_context)
# Build the index


In [5]:
# You already built: `nodes = [...]`
doc_names = sorted({ n.metadata.get("doc_name", "Unknown document") for n in nodes })
print("Documents in memory:")
for name in doc_names:
    print("-", name)



Documents in memory:
- 1 Alzheimer s Dementia - 2011 - Jack - Introduction to the recommendations from the National Institute on Aging‐Alzheimer s.pdf
- 2 Alzheimer s Dementia - 2011 - McKhann - The diagnosis of dementia due to Alzheimer s disease Recommendations from the.pdf
- 3 Alzheimer s Dementia - 2011 - Albert - The diagnosis of mild cognitive impairment due to Alzheimer s disease .pdf
- 4 Alzheimer s Dementia - 2011 - Sperling - Toward defining the preclinical stages of Alzheimer s disease Recommendations.pdf
- Clinical_Significance.json
- DGN Guidelines Diagnosis.pdf
- MRI for diagnosing dementia - update.pdf


In [ ]:
prompt="""
// refer prompts in the part 2 sectopn of this folder -https://github.com/DhanushBabu18/MRIs-to-Radiological-Dementia-Reports/tree/main/Prompts
"""

query_engine = index.as_query_engine()
response = query_engine.query(prompt)
print(response)
